# Salience-Sensitive Image Generation

Generates 50 images per prompt for each of:
1. DDPM baseline
2. ScoreNormPhi
3. DiversityPhi
4. ScoreAlignmentPhi

Prompts: llama, wolf, monkey, butterfly

Outputs saved to `../outputs/sd_experiment/<method>/<prompt>/`

In [2]:
import torch
from pathlib import Path
from diffusers import StableDiffusionPipeline

from sd.pipeline import SalienceGradSDPipeline
from sd.phi_sd import ScoreNormPhiSD, DiversityPhiSD, ScoreAlignmentPhiSD

DEVICE     = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
MODEL_ID   = "runwayml/stable-diffusion-v1-5"
OUTPUT_DIR = Path("../outputs/sd_experiment")
PROMPTS    = ["a green apple on a brown table", "a wooden chair in a blue room", "a wolf in the woods", "a piece of toast on a plate"]

NUM_IMAGES = 50
NUM_STEPS  = 500
CFG_SCALE  = 5.0
SAL_SCALE  = 1.0
BATCH_SIZE = 8
SEED       = 2024

print(f"Device: {DEVICE}")
print(f"Output dir: {OUTPUT_DIR}")

Device: mps
Output dir: ../outputs/sd_experiment


In [3]:
# ---------------------------------------------------------------------------
# Load salience pipeline and generate null embeddings
# ---------------------------------------------------------------------------
pipe = SalienceGradSDPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
).to(DEVICE)
pipe.set_progress_bar_config(disable=False)

null_tokens = pipe.tokenizer(
    [""],
    padding="max_length",
    max_length=pipe.tokenizer.model_max_length,
    return_tensors="pt",
).input_ids.to(DEVICE)

with torch.no_grad():
    null_embeds = pipe.text_encoder(null_tokens).last_hidden_state

print(f"Pipeline loaded. Null embeds shape: {null_embeds.shape}")

Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 17.90it/s]
You have disabled the safety checker for <class 'sd.pipeline.SalienceGradSDPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Pipeline loaded. Null embeds shape: torch.Size([1, 77, 768])


In [4]:
# ---------------------------------------------------------------------------
# Helper: generate N images for a prompt in batches
# ---------------------------------------------------------------------------
def generate_images(pipe, prompt, save_dir, n=50, batch_size=8, seed=2024):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    generated = 0
    batch_idx = 0
    while generated < n:
        current_batch = min(batch_size, n - generated)
        generator = torch.Generator(device=DEVICE).manual_seed(seed + batch_idx)
        pipe(
            prompt=prompt,
            num_images_per_prompt=current_batch,
            num_inference_steps=NUM_STEPS,
            guidance_scale=CFG_SCALE,
            generator=generator,
            save_dir=save_dir,
            offset=generated,
        )
        generated += current_batch
        batch_idx += 1
        print(f"    {generated}/{n}")

In [ ]:
import torch
import gc
import traceback
from pathlib import Path
from sd.pipeline import SalienceGradSDPipeline
from sd.phi_sd import ScoreAlignmentPhiSD

DEVICE     = "mps"
MODEL_ID   = "runwayml/stable-diffusion-v1-5"
OUTPUT_DIR = Path("../outputs/sd_preexperiment/scorealignment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pipe = SalienceGradSDPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    safety_checker=None,
).to(DEVICE)

null_tokens = pipe.tokenizer(
    [""],
    padding="max_length",
    max_length=pipe.tokenizer.model_max_length,
    return_tensors="pt",
).input_ids.to(DEVICE)

with torch.no_grad():
    null_embeds = pipe.text_encoder(null_tokens).last_hidden_state

phi_align = ScoreAlignmentPhiSD(conditional=False)
pipe.setup_phi(phi_align, context={"null_embeds": null_embeds})
pipe.set_salience_scale(1.0)
pipe.set_guidance_frequency(1)

# Monkey-patch AFTER pipe is defined
original_compute = pipe._compute_salience_gradient

def debug_compute(latents, t, context):
    try:
        return original_compute(latents, t, context)
    except Exception as e:
        print(f"Error at t={t}:")
        traceback.print_exc()
        raise

pipe._compute_salience_gradient = debug_compute

generator = torch.Generator(device=DEVICE).manual_seed(2024)
images = pipe(
    prompt="a red apple on a white table",
    num_images_per_prompt=2,
    num_inference_steps=50,
    guidance_scale=5.0,
    generator=generator,
).images

for idx, img in enumerate(images):
    img.save(OUTPUT_DIR / f"{idx}.png")

print("Done.")

/Users/keesvanhemmen/PycharmProjects/SalienceSensitiveDiffusion/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/keesvanhemmen/PycharmProjects/SalienceSensitiveDiffusion/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/keesvanhemmen/PycharmProjects/SalienceSensitiveDiffusion/.venv/lib/python3.9/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 57.56it/s]
You have disabled the safety checker for <class 'sd.pipeline.SalienceGra

In [ ]:
# ---------------------------------------------------------------------------
# Pre-experiment: sanity check salience guidance on SD
# ---------------------------------------------------------------------------
import torch
import gc
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'text.color': 'black',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'axes.edgecolor': 'black',
    'legend.labelcolor': 'black',
    'legend.facecolor': 'white',
})

from diffusers import StableDiffusionPipeline
from sd.pipeline import SalienceGradSDPipeline
from sd.phi_sd import ScoreNormPhiSD, DiversityPhiSD, ScoreAlignmentPhiSD

DEVICE     = "mps" if torch.backends.mps.is_available() else "cpu"
MODEL_ID   = "runwayml/stable-diffusion-v1-5"
OUTPUT_DIR = Path("../outputs/sd_preexperiment")
PROMPT     = "a red apple on a white table"

NUM_IMAGES            = 4
NUM_STEPS             = 50
CFG_SCALE             = 5.0
SAL_SCALE             = 1.0
BATCH_SIZE            = 4
ALIGNMENT_BATCH_SIZE  = 2   # ScoreAlignmentPhi uses 2N UNet passes, needs less memory
SEED                  = 2024

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def clear_memory():
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()

def generate_batched(pipe, prompt, n, batch_size, seed):
    images = []
    batch_idx = 0
    generated = 0
    while generated < n:
        current_batch = min(batch_size, n - generated)
        generator = torch.Generator(device=DEVICE).manual_seed(seed + batch_idx)
        result = pipe(
            prompt=prompt,
            num_images_per_prompt=current_batch,
            num_inference_steps=NUM_STEPS,
            guidance_scale=CFG_SCALE,
            generator=generator,
        ).images
        images.extend(result)
        generated += current_batch
        batch_idx += 1
    return images

results = {}

# ---------------------------------------------------------------------------
# 1. Baseline
# ---------------------------------------------------------------------------
print("1/4 Baseline...")
clear_memory()

baseline_pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    safety_checker=None,
).to(DEVICE)
baseline_pipe.set_progress_bar_config(disable=False)

results["baseline"] = generate_batched(baseline_pipe, PROMPT, NUM_IMAGES, BATCH_SIZE, SEED)

del baseline_pipe
clear_memory()
print("Baseline done.")

# ---------------------------------------------------------------------------
# 2. DiversityPhi
# ---------------------------------------------------------------------------
print("\n2/4 DiversityPhi...")
clear_memory()

pipe = SalienceGradSDPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    safety_checker=None,
).to(DEVICE)
pipe.set_progress_bar_config(disable=False)

null_tokens = pipe.tokenizer(
    [""],
    padding="max_length",
    max_length=pipe.tokenizer.model_max_length,
    return_tensors="pt",
).input_ids.to(DEVICE)

with torch.no_grad():
    null_embeds = pipe.text_encoder(null_tokens).last_hidden_state

phi_div = DiversityPhiSD()
pipe.setup_phi(phi_div, context={})
pipe.set_salience_scale(SAL_SCALE)
pipe.set_guidance_frequency(1)

results["diversity"] = generate_batched(pipe, PROMPT, NUM_IMAGES, BATCH_SIZE, SEED)

del pipe
clear_memory()
print("DiversityPhi done.")

# ---------------------------------------------------------------------------
# 3. ScoreAlignmentPhi
# ---------------------------------------------------------------------------
print("\n3/4 ScoreAlignmentPhi...")
clear_memory()

pipe = SalienceGradSDPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    safety_checker=None,
).to(DEVICE)
pipe.set_progress_bar_config(disable=False)

null_tokens = pipe.tokenizer(
    [""],
    padding="max_length",
    max_length=pipe.tokenizer.model_max_length,
    return_tensors="pt",
).input_ids.to(DEVICE)

with torch.no_grad():
    null_embeds = pipe.text_encoder(null_tokens).last_hidden_state

phi_align = ScoreAlignmentPhiSD(conditional=False)
pipe.setup_phi(phi_align, context={"null_embeds": null_embeds})
pipe.set_salience_scale(SAL_SCALE)
pipe.set_guidance_frequency(1)

results["scorealignment"] = generate_batched(
    pipe, PROMPT, NUM_IMAGES, ALIGNMENT_BATCH_SIZE, SEED
)

del pipe
clear_memory()
print("ScoreAlignmentPhi done.")

# ---------------------------------------------------------------------------
# Display results
# ---------------------------------------------------------------------------
methods = ["baseline", "diversity", "scorealignment"]
labels  = ["DDPM Baseline", "DiversityPhi", "ScoreAlignmentPhi"]

fig, axes = plt.subplots(
    len(methods), NUM_IMAGES,
    figsize=(NUM_IMAGES * 3, len(methods) * 3),
    facecolor="white",
)

for row, (method, label) in enumerate(zip(methods, labels)):
    for col, img in enumerate(results[method]):
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
        axes[row, col].set_facecolor("white")
    axes[row, 0].set_ylabel(label, fontsize=11, color="black")

    method_dir = OUTPUT_DIR / method
    method_dir.mkdir(parents=True, exist_ok=True)
    for idx, img in enumerate(results[method]):
        img.save(method_dir / f"{idx}.png")

fig.suptitle(f'"{PROMPT}" — salience scale {SAL_SCALE}, {NUM_STEPS} steps',
             fontsize=13, color="black")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "preexperiment_grid.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved to {OUTPUT_DIR / 'preexperiment_grid.png'}")

1/4 Baseline...


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 68.52it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
100%|██████████| 50/50 [01:05<00:00,  1.31s/it]


Baseline done.

2/4 DiversityPhi...


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 55.29it/s]
You have disabled the safety checker for <class 'sd.pipeline.SalienceGradSDPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
/Users/keesvanhemmen/PycharmProjects/SalienceSensitiveDiffusion/.venv/lib/python3.9/site-packages/diffusers/pipelines/stable_diffusion/pipeline_stable_diffusion.py:313: FutureWarning: `_encode_prompt()` is deprecated and it will be removed in a future version. Use `encode_prompt()` instead. Also, be aware that the output

DiversityPhi done.

3/4 ScoreAlignmentPhi...


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 59.36it/s]
You have disabled the safety checker for <class 'sd.pipeline.SalienceGradSDPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
  0%|          | 0/50 [00:00<?, ?it/s]

## 1. DDPM Baseline

In [4]:
print("=== DDPM Baseline ===")

baseline_pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
).to(DEVICE)
baseline_pipe.set_progress_bar_config(disable=False)

for prompt in PROMPTS:
    print(f"\nPrompt: {prompt}")
    save_dir = OUTPUT_DIR / "baseline" / prompt
    save_dir.mkdir(parents=True, exist_ok=True)
    generated = 0
    batch_idx = 0
    while generated < NUM_IMAGES:
        current_batch = min(BATCH_SIZE, NUM_IMAGES - generated)
        generator = torch.Generator(device=DEVICE).manual_seed(SEED + batch_idx)
        images = baseline_pipe(
            prompt=prompt,
            num_images_per_prompt=current_batch,
            num_inference_steps=NUM_STEPS,
            guidance_scale=CFG_SCALE,
            generator=generator,
        ).images
        for idx, img in enumerate(images):
            img.save(save_dir / f"{generated + idx}.png")
        generated += current_batch
        batch_idx += 1
        print(f"  {generated}/{NUM_IMAGES}")

del baseline_pipe
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print("\nBaseline done.")

=== DDPM Baseline ===


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 21.88it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .



Prompt: llama


100%|██████████| 500/500 [10:31<00:00,  1.26s/it]


  8/50


...

  40/50


100%|██████████| 500/500 [2:45:53<00:00, 19.91s/it]    


  48/50


100%|██████████| 500/500 [27:59<00:00,  3.36s/it]    


  50/50

Baseline done.


## 2. ScoreNormPhi

In [ ]:
print("=== ScoreNormPhi ===")

phi_norm = ScoreNormPhiSD(conditional=False)
pipe.setup_phi(phi_norm, context={"null_embeds": null_embeds})
pipe.set_salience_scale(SAL_SCALE)
pipe.set_guidance_frequency(1)

for prompt in PROMPTS:
    print(f"\nPrompt: {prompt}")
    generate_images(
        pipe, prompt,
        save_dir=OUTPUT_DIR / "scorenorm" / prompt,
        n=NUM_IMAGES, batch_size=BATCH_SIZE, seed=SEED,
    )

print("\nScoreNormPhi done.")

## 3. DiversityPhi

In [ ]:
print("=== DiversityPhi ===")

phi_div = DiversityPhiSD()
pipe.setup_phi(phi_div, context={})
pipe.set_salience_scale(SAL_SCALE)
pipe.set_guidance_frequency(1)

for prompt in PROMPTS:
    print(f"\nPrompt: {prompt}")
    generate_images(
        pipe, prompt,
        save_dir=OUTPUT_DIR / "diversity" / prompt,
        n=NUM_IMAGES, batch_size=BATCH_SIZE, seed=SEED,
    )

print("\nDiversityPhi done.")

## 4. ScoreAlignmentPhi

In [ ]:
print("=== ScoreAlignmentPhi ===")

phi_align = ScoreAlignmentPhiSD(conditional=False)
pipe.setup_phi(phi_align, context={"null_embeds": null_embeds})
pipe.set_salience_scale(SAL_SCALE)
pipe.set_guidance_frequency(1)

for prompt in PROMPTS:
    print(f"\nPrompt: {prompt}")
    generate_images(
        pipe, prompt,
        save_dir=OUTPUT_DIR / "scorealignment" / prompt,
        n=NUM_IMAGES, batch_size=BATCH_SIZE, seed=SEED,
    )

print("\nScoreAlignmentPhi done.")

## Done

All images saved to `../outputs/sd_experiment/`. Run the evaluation notebook next.